# Running the test cases

Open this notebook from `Conformaty` or from `Conformaty/test_cases`.

The source modules are in `src/` (`cli.py`, `chemistry`, `gaussian`, `fileio`, `numbering`, `optimization`, `pipelines`). This notebook calls `main.py`, which loads them from there. Paths below are relative to `Conformaty`.

These cases are the ones where atom correspondence is ambiguous. Each section uses a different part of the workflow to choose among those alternatives. The folders hold the starting structures. Results are written into that case's `output` folder.

In [1]:
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
if (here / "main.py").is_file():
    ROOT = here
elif (here.parent / "main.py").is_file():
    ROOT = here.parent
else:
    raise SystemExit("Open this notebook from Conformaty or from Conformaty/test_cases")

def run(args):
    print("python main.py", " ".join(args), flush=True)
    subprocess.run([sys.executable, str(ROOT / "main.py"), *args], cwd=ROOT, check=True)


## 01 Structural alignment

Goal: superimpose the cyclohexane boat on the chair by a rigid move only. The internal geometry stays fixed.

RMSD gives the smallest overall score, but some matched carbons stay apart. The combined loss (sigmoid plus a linear distance term, $\sigma = 20$) instead rewards atoms that land almost on top of each other, even if another atom, such as carbon 4, stays farther away.

In [2]:
case = "test_cases/01_structural_alignment"
# Combined loss: more carbons in near-exact overlap.
run([
    "align", "-t", f"{case}/cyclohexane_chair.xyz", "-q", f"{case}/cyclohexane_boat.xyz",
    "--loss-type", "mixed", "--sigma", "20", "-o", f"{case}/output/mixed",
])
# RMSD: smallest overall squared distance; some matched atoms remain apart.
run([
    "align", "-t", f"{case}/cyclohexane_chair.xyz", "-q", f"{case}/cyclohexane_boat.xyz",
    "--loss-type", "rmsd", "-o", f"{case}/output/rmse",
])


python main.py align -t test_cases/01_structural_alignment/cyclohexane_chair.xyz -q test_cases/01_structural_alignment/cyclohexane_boat.xyz --loss-type mixed --sigma 20 -o test_cases/01_structural_alignment/output/mixed
python main.py align -t test_cases/01_structural_alignment/cyclohexane_chair.xyz -q test_cases/01_structural_alignment/cyclohexane_boat.xyz --loss-type rmsd -o test_cases/01_structural_alignment/output/rmse


## 02 Prochirality

Goal: number cyclohexanol onto the cyclohexane boat without swapping prochiral groups.

Two hydrogens on a carbon can be equivalent by connectivity and still point different ways in space. One assignment keeps that orientation. Exchanging them is the same on a graph and the wrong correspondence in 3D. The same check applies to any equivalent groups on a matched atom, not only hydrogens.

In [ ]:
case = "test_cases/02_prochirality"
# Keep the 3D order of equivalent substituents, not only their connectivity.
run([
    "pair", "-t", f"{case}/cyclohexanol.xyz", "-q", f"{case}/cyclohexane_boat.xyz",
    "-o", f"{case}/output",
])


## 03 Ring numbering

Goal: show that one pair, cyclohexane chair and methyl-cyclofurane, can be numbered in more than one defensible way. The user chooses which rule wins.

Ring completion places the oxygen on reference carbon 5 so the rings correspond. MCS extension places the methyl carbon there. Overlap mapping also places the oxygen on carbon 5, because that is where it sits after alignment.

In [ ]:
case = "test_cases/03_ring_numbering"
# complete_rings: O -> C5. extend_mcs: methyl C -> C5. coords_proximity_any: O -> C5 by overlap.
for stage in ("complete_rings", "extend_mcs", "coords_proximity_any"):
    run([
        "pair", "-t", f"{case}/stage_order_ref.xyz", "-q", f"{case}/stage_order_query.xyz",
        "--stages", stage, "-o", f"{case}/output/{stage}",
    ])


## 04 Radicals

Goal: keep the radical on the intended ring carbon. XYZ has no bonds, so the radical center is inferred from distances and valence. Another atom, often a neighboring oxygen, can fit that pattern, and the match then fails. An anchor forces the radical-bearing carbon to stay paired, which is enough to renumber the set without labeling every atom by hand.

The structures are from Zhang et al., *A neural network model informs the total synthesis of clovane sesquiterpenoids* (Nature Synthesis, 2023). The first run is `ML2` onto `ML275` with no anchor. The second pins that carbon. The third does the same for the whole set, using one anchor per molecule from the spreadsheet, with `ML112` as the reference.

In [ ]:
case = "test_cases/04_radicals"
# No anchor: the inferred radical can move onto a neighboring oxygen.
run([
    "pair", "-t", f"{case}/ML2.xyz", "-q", f"{case}/ML275.xyz",
    "--match-charge", "-o", f"{case}/output/pair_no_anchor",
])
# Anchor the radical-bearing carbon so the two radicals correspond.
run([
    "pair", "-t", f"{case}/ML2.xyz", "-q", f"{case}/ML275.xyz",
    "--match-charge", "--anchor-ref", "1", "--anchor-query", "4",
    "-o", f"{case}/output/pair_with_anchor",
])
# Same idea for the full set: one anchored atom per molecule.
run([
    "batch", "-f", case, "-r", f"{case}/ML112.xyz",
    "--anchors-file", f"{case}/Mol_RadNum.xlsx", "--match-charge",
    "-o", f"{case}/output/batch_anchored",
])


## 05 Mechanism numbering

Goal: give every structure along one pathway the same atom numbers. The geometries are from Formica et al., *Catalytic enantioselective nucleophilic desymmetrization of phosphonate esters* (Nature Chemistry, 2023): reactant, intermediates, transition states, and product.

The batch follows the phosphonate substrate, `SM.xyz`. The first three pairs do the same for the intermediate, product, and one transition state. The last two pairs instead follow both the phenyl ester and the aryl alcohol (`SM_ArOH`). Matching several components at once is possible, and it gets expensive as the structures grow.

In [ ]:
case = "test_cases/05_mechanism"
# Whole pathway numbered onto the phosphonate substrate.
run(["batch", "-f", case, "-r", f"{case}/SM.xyz", "-o", f"{case}/output/batch_vs_SM"])
# SM_* follows that substrate. SM_ArOH_* also follows the aryl alcohol.
for name, target, query in (
    ("SM_INT_R", "SM.xyz", "INT-(R).xyz"),
    ("SM_P", "SM.xyz", "P.xyz"),
    ("SM_TS1_R", "SM.xyz", "TS1-(R).xyz"),
    ("SM_ArOH_INT_R", "SM_ArOH.xyz", "INT-(R).xyz"),
    ("SM_ArOH_TS1_R", "SM_ArOH.xyz", "TS1-(R).xyz"),
):
    run([
        "pair", "-t", f"{case}/{target}", "-q", f"{case}/{query}",
        "-o", f"{case}/output/{name}",
    ])
